In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/keshavsharmagg/faiss-index-laws-de/index.faiss
/kaggle/input/datasets/keshavsharmagg/faiss-index-laws-de/index.pkl
/kaggle/input/datasets/keshavsharmagg/train-csv/train.csv
/kaggle/input/datasets/keshavsharmagg/laws-de/laws_de.csv
/kaggle/input/datasets/keshavsharmagg/reranker-swiss-law/config.json
/kaggle/input/datasets/keshavsharmagg/reranker-swiss-law/tokenizer.json
/kaggle/input/datasets/keshavsharmagg/reranker-swiss-law/tokenizer_config.json
/kaggle/input/datasets/keshavsharmagg/reranker-swiss-law/CEBinaryClassificationEvaluator_swiss-law-reranker-eval_results.csv
/kaggle/input/datasets/keshavsharmagg/reranker-swiss-law/model.safetensors
/kaggle/input/datasets/keshavsharmagg/reranker-swiss-law/special_tokens_map.json


In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Authenticate
login(token=hf_token)

In [4]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    '/kaggle/input/datasets/keshavsharmagg/reranker-swiss-law',
    num_labels=1,
    device='cuda'
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
!pip install langchain-community langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, whi

In [6]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.0 MB/s eta 0:00:00:00:0100:01


In [7]:
import json
import re
import time as _time
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, List, Optional, Sequence, Union

import pandas as pd
import requests
from requests.exceptions import RequestException
from sentence_transformers import CrossEncoder
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()
print('Imports OK')

Imports OK


In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

vectorstore = FAISS.load_local(
    '/kaggle/input/datasets/keshavsharmagg/faiss-index-laws-de',
    embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 100},
)

train_csv = pd.read_csv('/kaggle/input/datasets/keshavsharmagg/train-csv/train.csv')
laws_de   = pd.read_csv('/kaggle/input/datasets/keshavsharmagg/laws-de/laws_de.csv')

print(f'train rows  : {len(train_csv)}')
print(f'laws_de rows: {len(laws_de)}')

/tmp/ipykernel_55/657855169.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

train rows  : 1139
laws_de rows: 175933


In [9]:
"""
Load the finetuned LoRA adapter locally and generate 3 search queries.

Usage:
    python inference_local.py
"""
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL   = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_PATH = "keshavsharma/llama-3.1-7b-swiss-law-adapter"   # downloaded via download_adapter.py

def load_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    model.eval()
    return model, tokenizer


def generate_queries(case_query: str, model, tokenizer, temperature: float = 0.4) -> list[str]:
    messages = [
        {
            "role": "system",
            "content": (
                "Du bist ein Rechtsrecherche-Spezialist für Schweizer Recht. "
                "Gegeben einen deutschen Rechtsfall, generiere genau 3 kurze deutsche "
                "Stichwortsuchanfragen, die die relevantesten Gesetzesartikel aus einer "
                "Vektordatenbank abrufen. "
                "Gib NUR die 3 Suchanfragen aus, eine pro Zeile, ohne Nummerierung, "
                "ohne Symbole, ohne weiteren Text."
            ),
        },
        {"role": "user", "content": case_query},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=124,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    queries = [q.strip() for q in generated.strip().split("\n") if q.strip()]
    return queries[:3]



In [10]:
print("Loading model...")
model, tokenizer = load_model()

# test_case = train_csv["query"][1]


Loading model...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

In [47]:
### finetuned llm generated queries
train_index = 3
search_queries = generate_queries(train_csv["query"][train_index], model, tokenizer, temperature=0.7)

for i, q in enumerate(search_queries, 1):
    print(f"  Query {i}: {q}")

## checking gold hits
train_row = train_csv.iloc[train_index]
gold_citations = [c.strip() for c in str(train_row["gold_citations"]).split(";") if c.strip()]
print("gold citations:", gold_citations)
gold_casefold = set(c.casefold() for c in gold_citations)

total_gold_hits = []

total_unique_retrieved = set()

for query in search_queries:
    print(f"\nSearch query: '{query}'")
    docs = vectorstore.similarity_search(query, k=500)
    retrieved_citations = [
        str(d.metadata.get("citation")).strip()
        for d in docs
        if d.metadata.get("citation") and str(d.metadata.get("citation")).strip()
    ]
    # print(f"Retrieved citations: {retrieved_citations}")

    gold_hits = [rc for rc in retrieved_citations if rc.casefold() in gold_casefold]
    print(f"Gold hits in retrieved: {gold_hits}")
    for hit in gold_hits:
        if hit not in total_gold_hits:
            total_gold_hits.append(hit)
    total_unique_retrieved.update(retrieved_citations)

print(f"\nTotal unique gold hits across all queries: {total_gold_hits.__len__()} out of {len(gold_citations)} gold citations")
print(f"Total unique retrieved citations across all queries: {len(total_unique_retrieved)}")

print(f"precision: {len(total_gold_hits) / len(total_unique_retrieved):.4f} | recall: {len(total_gold_hits) / len(gold_citations):.4f}")
print(f"f1 score: {2 * (len(total_gold_hits) / len(total_unique_retrieved)) * (len(total_gold_hits) / len(gold_citations)) / ((len(total_gold_hits) / len(total_unique_retrieved)) + (len(total_gold_hits) / len(gold_citations))):.4f}")

  Query 1: parteien vereinbaren zuständigkeit schiedsinstitution entscheidet unabhängig hauptsache
  Query 2: zuständigkeit schiedsinstitution entscheidet unabhängig hauptsache kann entscheid
  Query 3: schiedsinstitution entscheidet unabhängig hauptsache kann entscheid protokolls artikel
gold citations: ['Art. 176 Abs. 1 IPRG', 'Art. 186 Abs. 1 IPRG', 'Art. 186 Abs. 3 IPRG', 'Art. 190 Abs. 3 IPRG']

Search query: 'parteien vereinbaren zuständigkeit schiedsinstitution entscheidet unabhängig hauptsache'
Gold hits in retrieved: ['Art. 186 Abs. 1 IPRG', 'Art. 186 Abs. 3 IPRG']

Search query: 'zuständigkeit schiedsinstitution entscheidet unabhängig hauptsache kann entscheid'
Gold hits in retrieved: ['Art. 186 Abs. 1 IPRG', 'Art. 186 Abs. 3 IPRG']

Search query: 'schiedsinstitution entscheidet unabhängig hauptsache kann entscheid protokolls artikel'
Gold hits in retrieved: ['Art. 186 Abs. 1 IPRG', 'Art. 186 Abs. 3 IPRG']

Total unique gold hits across all queries: 2 out of 4 gold citations


## now building dataset for reranker

In [ ]:
import json
import os
import gc
import torch
from sentence_transformers import InputExample
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
SAVE_PATH      = "reranker_samples.json"
ERROR_LOG_PATH = "reranker_errors.json"
K_RETRIEVE     = 200   # reduced from 1000 — T4 can't hold 1000 doc embeddings per query × 3 queries
NEG_RATIO      = 3
RESUME_FROM    = 0     # will be auto-set if save file exists

# ── Resume logic ──────────────────────────────────────────────────────────────
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH) as f:
        saved = json.load(f)
    RESUME_FROM = saved["last_completed_idx"] + 1
    raw_samples = saved["samples"]   # list of dicts
    print(f"Resuming from row {RESUME_FROM}, already have {len(raw_samples)} samples")
else:
    raw_samples = []
    RESUME_FROM = 0
    print("Starting fresh")

error_log = []

def save_checkpoint(idx):
    with open(SAVE_PATH, "w") as f:
        json.dump({"last_completed_idx": idx, "samples": raw_samples}, f)

def flush_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# ── Main loop ─────────────────────────────────────────────────────────────────
for idx, row in tqdm(train_csv.iterrows(), total=len(train_csv)):
    if idx < RESUME_FROM:
        continue

    query     = row['query']
    gold_cits = [c.strip() for c in row['gold_citations'].split(';') if c.strip()]

    try:
        # ── Step 1: Generate queries with Llama ──────────────────────────────
        flush_gpu()
        llama_queries = generate_queries(query, model, tokenizer, temperature=0.7)
        flush_gpu()

        # ── Step 2: Retrieve candidates (smaller k to avoid OOM) ─────────────
        candidate_docs = []
        seen_cits = set()

        for lq in llama_queries:
            try:
                docs = vectorstore.similarity_search(lq, k=K_RETRIEVE)
            except Exception as e:
                print(f"  [Row {idx}] Retrieval failed for query '{lq[:40]}': {e}")
                flush_gpu()
                continue

            for d in docs:
                cit = d.metadata.get('citation', '').strip()
                if cit and cit not in seen_cits:
                    candidate_docs.append(d)
                    seen_cits.add(cit)

            flush_gpu()  # clear after each query retrieval

        if not candidate_docs:
            print(f"  [Row {idx}] No candidates retrieved — skipping")
            error_log.append({"idx": idx, "reason": "no candidates retrieved"})
            save_checkpoint(idx)
            continue

        # ── Step 3: Build positives ───────────────────────────────────────────
        row_samples = []

        for cit in gold_cits:
            match = laws_de[laws_de['citation'] == cit]['text'].values
            if len(match):
                row_samples.append({
                    "texts": [query, match[0][:512]],
                    "label": 1.0
                })

        if not row_samples:
            print(f"  [Row {idx}] No positive matches in laws_de — skipping")
            error_log.append({"idx": idx, "reason": "no positives in laws_de"})
            save_checkpoint(idx)
            continue

        # ── Step 4: Hard negatives ────────────────────────────────────────────
        non_gold = [
            d for d in candidate_docs
            if d.metadata.get('citation', '').strip() not in gold_cits
        ]
        n_neg = len(row_samples) * NEG_RATIO  # positives × 3
        for doc in non_gold[:n_neg]:
            row_samples.append({
                "texts": [query, doc.page_content[:512]],
                "label": 0.0
            })

        raw_samples.extend(row_samples)

        # ── Checkpoint every 10 rows ──────────────────────────────────────────
        if idx % 10 == 0:
            save_checkpoint(idx)
            pos_so_far = sum(1 for s in raw_samples if s["label"] == 1.0)
            neg_so_far = sum(1 for s in raw_samples if s["label"] == 0.0)
            print(f"  [Row {idx}] checkpoint | total={len(raw_samples)} | pos={pos_so_far} | neg={neg_so_far}")

    except torch.cuda.OutOfMemoryError:
        flush_gpu()
        print(f"  [Row {idx}] OOM — skipping this row")
        error_log.append({"idx": idx, "reason": "OOM"})
        save_checkpoint(idx)   # save progress before continuing
        continue

    except Exception as e:
        print(f"  [Row {idx}] Unexpected error: {type(e).__name__}: {e} — skipping")
        error_log.append({"idx": idx, "reason": f"{type(e).__name__}: {str(e)}"})
        save_checkpoint(idx)
        continue

# ── Final save ────────────────────────────────────────────────────────────────
save_checkpoint(len(train_csv) - 1)

with open(ERROR_LOG_PATH, "w") as f:
    json.dump(error_log, f, indent=2)

print(f"\nDone. Total samples: {len(raw_samples)}")
pos = sum(1 for s in raw_samples if s["label"] == 1.0)
neg = sum(1 for s in raw_samples if s["label"] == 0.0)
print(f"Positives: {pos} | Negatives: {neg} | Ratio: {neg/pos:.1f}:1")
print(f"Skipped rows: {len(error_log)} — see {ERROR_LOG_PATH}")

# ── Convert to InputExamples for training ─────────────────────────────────────
reranker_samples = [
    InputExample(texts=s["texts"], label=s["label"])
    for s in raw_samples
]

Starting fresh



  0%|          | 1/1139 [00:19<6:02:06, 19.09s/it]

  [Row 0] checkpoint | total=8 | pos=2 | neg=6



  0%|          | 3/1139 [00:56<5:53:08, 18.65s/it]

  [Row 2] No positive matches in laws_de — skipping


...


  3%|▎         | 30/1139 [09:23<5:55:22, 19.23s/it]

  [Row 29] No positive matches in laws_de — skipping



  3%|▎         | 31/1139 [09:41<5:50:31, 18.98s/it]

  [Row 30] checkpoint | total=324 | pos=81 | neg=243



  3%|▎         | 35/1139 [10:56<5:43:36, 18.67s/it]

### trying both combinations

In [ ]:
def rerank_with_threshold(original_case_query, queries, reranker, score_threshold=0.3, K_RETRIEVE=1000, max_citations=100):
    # Collect all candidates across all queries
    all_docs = {}
    prererank_cits = []
    for q in queries:
        for doc in vectorstore.similarity_search(q, k=K_RETRIEVE):
            cit = doc.metadata.get('citation', '')
            if cit not in all_docs:
                all_docs[cit] = doc
                prererank_cits.append(cit)
    
    candidates = list(all_docs.values())
    
    # Rerank against the original case query (not the keyword queries)
    # This is important — the case query is richer context for the reranker
    pairs  = [(original_case_query, doc.page_content) for doc in candidates]
    scores = reranker.predict(pairs)
    
    # Filter by threshold instead of hard top-k
    results = [
        (doc, score) for doc, score in zip(candidates, scores)
        if score > score_threshold
    ]
    results.sort(key=lambda x: x[1], reverse=True)
    return [(doc.metadata['citation'],score) for doc, score in results[:max_citations]], prererank_cits


def get_gold_cits(cits_fetched, gold_cits, _type="pre"):
    gold_found = []
    for cit in cits_fetched:
        if _type=="post":
            score = cit[1]
            cit = cit[0]
            if (cit in gold_cits):
                if (cit in gold_found):
                    continue
                print("post rerank citation with score: ", cit, score)
                gold_found.append(cit)
        else:
            if (cit in gold_cits):
                if (cit in gold_found):
                    continue
                gold_found.append(cit)
    
    return gold_found
    
    

In [44]:
### finetuned llm generated queries
train_index = 3
original_case_query = train_csv["query"][train_index]
search_queries = generate_queries(original_case_query, model, tokenizer, temperature=0.7)

for i, q in enumerate(search_queries, 1):
    print(f"  Query {i}: {q}")


  Query 1: darbietung eigener klage verletzung urteilsform bedroht erscheint
  Query 2: erstinstanz richter heftigungsbegehren jeder klage erhebt eintragung
  Query 3: richter heftigungsbegehren jeder klage erhebt eintragung löschen


In [48]:
search_queries

['parteien vereinbaren zuständigkeit schiedsinstitution entscheidet unabhängig hauptsache',
 'zuständigkeit schiedsinstitution entscheidet unabhängig hauptsache kann entscheid',
 'schiedsinstitution entscheidet unabhängig hauptsache kann entscheid protokolls artikel']

In [49]:
original_case_query

'Die Alpha Consulting AG kann nun ihr Grundstück neu direkt an die soeben erstellte Kana lisationsleitung in der angrenzenden neuen Quartierstrasse anschliessen, so dass sie die von der Grunddienstbarkeit erfasste Abwasserleitung (vgl. Sachverhalt Teil 1) ausser Betrieb nimmt und ihr Grundstück jetzt an die Leitung in der Quartierstrasse anschliesst. Georg (G), der Grundbuchverwalter des Grundbuchkreises, in dem die Grundstücke 1 und 2 liegen, wohnt ganz in der Nähe und beobachtet auf seinem täglichen Weg zur Arbeit die Installation der neuen Kanalisationsleitung in der Quartierstrasse. Einige Tage später liest er zudem in der Zeitung, dass sämtliche Liegenschaften, die an die Quartierstrasse grenzen, neu an die in der Quartierstrasse befindliche Kanalisationsleitung angeschlossen worden seien. Daher fasst Georg den Entschluss, zur Entlastung des Grundbuchs die bisher eingetra gene Leitungsdienstbarkeit von Amtes wegen zu löschen, und setzt sein Vorhaben sofort in die Tat um. \nSollte 

In [ ]:
postrerank_cits, prererank_cits = rerank_with_threshold(
    original_case_query, 
    search_queries, 
    reranker,
    score_threshold=0.9,
    K_RETRIEVE=1000,
    max_citations=300
)
ground_citations = train_csv["gold_citations"][train_index].split(";")
epsilon = 1e-9

print("ground truth citations count: ", ground_citations.__len__())

## pre rerank cits
print("pre rerank cits fetched: ", prererank_cits.__len__())
prererank_gold_cits_found = get_gold_cits(prererank_cits, ground_citations)

prererank_recall = prererank_gold_cits_found.__len__() / (ground_citations.__len__() + epsilon)
prererank_precision = prererank_gold_cits_found.__len__() / (prererank_cits.__len__() + epsilon)

print("prererank gold found: ", prererank_gold_cits_found.__len__())
print("prererank recall: ", prererank_recall)
print("prererank precision: ", prererank_precision)
print("prererank F1: ", 2 * (prererank_precision * prererank_recall)/ (prererank_precision + prererank_recall + epsilon))
print(prererank_gold_cits_found)
print("-"*30)

## post rerank cits
print("post rerank cits fetched: ", postrerank_cits.__len__())
postrerank_gold_cits_found = get_gold_cits(postrerank_cits, ground_citations, _type="post")

postrerank_recall = postrerank_gold_cits_found.__len__() / (ground_citations.__len__() + epsilon)
postrerank_precision = postrerank_gold_cits_found.__len__() / (postrerank_cits.__len__() + epsilon)

print("postrerank gold found: ", postrerank_gold_cits_found.__len__())
print("postrerank recall: ", postrerank_recall)
print("postrerank precision: ", postrerank_precision)
print("postrerank F1: ", 2 * (postrerank_precision * postrerank_recall)/ (postrerank_precision + postrerank_recall + epsilon))
print(postrerank_gold_cits_found)
print("-"*30)

ground truth citations count:  4
pre rerank cits fetched:  1067
prererank gold found:  1
prererank recall:  0.2499999999375
prererank precision:  0.0009372071227732547
prererank F1:  0.0018674136246742705
['Art. 186 Abs. 3 IPRG']
------------------------------
post rerank cits fetched:  50
postrerank gold found:  1
postrerank recall:  0.2499999999375
postrerank precision:  0.0199999999996
postrerank F1:  0.03703703689849108
['Art. 186 Abs. 3 IPRG']
------------------------------


In [28]:
train_csv["gold_citations"][train_index].split(";")

['Art. 176 Abs. 1 IPRG',
 'Art. 186 Abs. 1 IPRG',
 'Art. 186 Abs. 3 IPRG',
 'Art. 190 Abs. 3 IPRG']

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as HFTokenizer

# ---------- Paths ----------
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
OUTPUT_DIR = Path("../output") if Path("../output").exists() else Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- EN -> DE Translation Model (best-first with fallback) ----------
TRANSLATION_MODEL_CANDIDATES = [
    "facebook/nllb-200-distilled-1.3B",
    "facebook/nllb-200-distilled-600M",
]

translator_model = None
translator_tokenizer = None
translation_model_id = None

for model_id in TRANSLATION_MODEL_CANDIDATES:
    try:
        translator_tokenizer = HFTokenizer.from_pretrained(model_id)
        translator_model = AutoModelForSeq2SeqLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        if not torch.cuda.is_available():
            translator_model.to("cpu")
        translation_model_id = model_id
        print(f"Loaded EN->DE translator: {translation_model_id}")
        break
    except Exception as exc:
        print(f"Failed to load {model_id}: {exc}")

if translator_model is None:
    raise RuntimeError("Could not load an EN->DE Hugging Face model.")

# ---------- Keep current pipeline behavior/config ----------
PIPELINE_CFG = {
    "temperature": 0.7,
    "score_threshold": 0.9,
    "K_RETRIEVE": 1000,
    "max_citations": 300,
}


def translate_en_to_de(texts, batch_size=8, max_new_tokens=256):
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]

    translator_tokenizer.src_lang = "eng_Latn"
    forced_bos_token_id = translator_tokenizer.convert_tokens_to_ids("deu_Latn")
    device = translator_model.device

    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = translator_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            generated_tokens = translator_model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=max_new_tokens,
                num_beams=4,
            )

        decoded = translator_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        out.extend([t.strip() for t in decoded])

    return out[0] if single_input else out


def predict_citations_for_german_query(query_de, cfg=PIPELINE_CFG):
    search_queries = generate_queries(
        query_de,
        model,
        tokenizer,
        temperature=cfg["temperature"],
    )

    if not search_queries:
        search_queries = [query_de]

    try:
        postrerank_cits, _ = rerank_with_threshold(
            original_case_query=query_de,
            queries=search_queries,
            reranker=reranker,
            score_threshold=cfg["score_threshold"],
            K_RETRIEVE=cfg["K_RETRIEVE"],
            max_citations=cfg["max_citations"],
        )
    except Exception as exc:
        print(f"[WARN] Rerank failed: {exc}")
        return search_queries, []

    deduped = []
    seen = set()
    for cit, _score in postrerank_cits:
        cit = str(cit).strip()
        if cit and cit not in seen:
            seen.add(cit)
            deduped.append(cit)

    return search_queries, deduped


print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"Translation model in use: {translation_model_id}")

In [ ]:
# ---------- TEST: Full submission generation ----------
test_path = DATA_DIR / "test.csv"
test_df = pd.read_csv(test_path)
print(f"Loaded test rows: {len(test_df)} from {test_path}")

# Translate all EN queries to DE first
translated_test_queries = translate_en_to_de(test_df["query"].astype(str).tolist(), batch_size=8)

test_predictions = []
for idx, row in enumerate(tqdm(test_df.itertuples(index=False), total=len(test_df), desc="Test inference")):
    query_de = translated_test_queries[idx]
    _search_queries, pred_cits = predict_citations_for_german_query(query_de, PIPELINE_CFG)

    test_predictions.append(
        {
            "query_id": row.query_id,
            "predicted_citations": ";".join(pred_cits),
        }
    )

submission_test_df = pd.DataFrame(test_predictions, columns=["query_id", "predicted_citations"])
test_output_path = OUTPUT_DIR / "submission_test_en2de.csv"
submission_test_df.to_csv(test_output_path, index=False)

print(f"Saved test submission to: {test_output_path}")
display(submission_test_df.head())

In [ ]:
# ---------- VAL: Full pipeline + metrics ----------
val_path = DATA_DIR / "val.csv"
val_df = pd.read_csv(val_path)
print(f"Loaded val rows: {len(val_df)} from {val_path}")

# Translate all EN queries to DE first
translated_val_queries = translate_en_to_de(val_df["query"].astype(str).tolist(), batch_size=8)

eps = 1e-12
micro_tp = 0
micro_fp = 0
micro_fn = 0

row_precisions = []
row_recalls = []
row_f1s = []

val_predictions = []
for idx, row in enumerate(tqdm(val_df.itertuples(index=False), total=len(val_df), desc="Val inference")):
    query_de = translated_val_queries[idx]
    _search_queries, pred_cits = predict_citations_for_german_query(query_de, PIPELINE_CFG)

    gold_cits = [c.strip() for c in str(row.gold_citations).split(";") if c.strip()]
    gold_set = set(gold_cits)
    pred_set = set(pred_cits)

    tp = len(pred_set & gold_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    micro_tp += tp
    micro_fp += fp
    micro_fn += fn

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)

    row_precisions.append(precision)
    row_recalls.append(recall)
    row_f1s.append(f1)

    val_predictions.append(
        {
            "query_id": row.query_id,
            "predicted_citations": ";".join(pred_cits),
            "gold_citations": ";".join(gold_cits),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
    )

micro_precision = micro_tp / (micro_tp + micro_fp + eps)
micro_recall = micro_tp / (micro_tp + micro_fn + eps)
micro_f1 = (2 * micro_precision * micro_recall) / (micro_precision + micro_recall + eps)

macro_precision = float(np.mean(row_precisions)) if row_precisions else 0.0
macro_recall = float(np.mean(row_recalls)) if row_recalls else 0.0
macro_f1 = float(np.mean(row_f1s)) if row_f1s else 0.0

val_predictions_df = pd.DataFrame(val_predictions)
val_output_path = OUTPUT_DIR / "val_predictions_en2de.csv"
val_predictions_df.to_csv(val_output_path, index=False)

val_metrics_df = pd.DataFrame(
    [
        {
            "split": "val_full",
            "averaging": "micro",
            "precision": micro_precision,
            "recall": micro_recall,
            "f1": micro_f1,
        },
        {
            "split": "val_full",
            "averaging": "macro",
            "precision": macro_precision,
            "recall": macro_recall,
            "f1": macro_f1,
        },
    ]
)

metrics_output_path = OUTPUT_DIR / "val_metrics_en2de.csv"
val_metrics_df.to_csv(metrics_output_path, index=False)

print("Validation metrics:")
display(val_metrics_df)
print(f"Saved val predictions to: {val_output_path}")
print(f"Saved val metrics to: {metrics_output_path}")